In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
osm = gpd.read_parquet('../../adatok/fix/osm_cimek_teljes.parquet')

In [3]:
gmaps = gpd.read_parquet('../../adatok/fix/teljes_ms_lekerdezes_tisztott.parquet')

In [4]:
len(osm)

623656

In [5]:
osm.head()

,id,geometry,addr:city,addr:postcode,addr:street,addr:housenumber
0,node/16947670,POINT (18.13766 46.14637),Orfű,7677,Dollár út,1
1,node/25546152,POINT (18.22533 46.06637),Pécs,7623,Indóház tér,1
2,node/33123162,POINT (17.55781 46.7501),Fonyód,8640,Ady Endre utca,2
3,node/33127897,POINT (18.04147 46.7341),Tab,8660,Kossuth Lajos utca,1
4,node/94743595,POINT (17.30675 46.28901),Somogyszob,7563,Keleti köz,1


In [6]:
len(gmaps)

2190747

In [7]:
gmaps.head()

,gid,utca,cim,telepules,iszam,orszag,geometry
0,0,Csalitos utca,6,Tököl,2316,Hungary,POINT (19.00889 47.28993)
1,1,Diana sétány,55,Tököl,2316,Hungary,POINT (19.01053 47.29057)
2,2,Diana sétány,71,Tököl,2316,Hungary,POINT (19.00921 47.28975)
3,3,Diana sétány,61,Tököl,2316,Hungary,POINT (19.00999 47.29026)
4,4,Diana sétány,147,Tököl,2316,Hungary,POINT (19.00071 47.28387)


---

In [8]:
print("egyedi értékek száma:", gmaps["orszag"].nunique(dropna=False))
print()
print(gmaps["orszag"].value_counts(dropna=False))

egyedi értékek száma: 3

orszag
Hungary    2190737
Austria          9
Romania          1
Name: count, dtype: int64


In [9]:
# gmaps ország cella kidobása de előtte minden nem Hungary sor törlése
gmaps = gmaps[gmaps["orszag"] == "Hungary"].copy()
gmaps = gmaps.drop(columns=["orszag"])
print("gmaps Hungary szűrés után:", len(gmaps))

gmaps Hungary szűrés után: 2190737


In [10]:
# id oszlopok törlés (egy közös id lesz majd)
osm = osm.drop(columns=["id"])
gmaps = gmaps.drop(columns=["gid"])

In [11]:
# oszlopok közös névre hozása
gmaps = gmaps.rename(columns={
    "telepules": "city",
    "iszam": "postcode",
    "utca": "street",
    "cim": "housenumber",
})

osm = osm.rename(columns={
    "addr:city": "city",
    "addr:postcode": "postcode",
    "addr:street": "street",
    "addr:housenumber": "housenumber",
})

In [12]:
# rögzítés hogy melyik datasetből jön a sor
osm["source"] = "osm"
gmaps["source"] = "gmaps"

In [13]:
match_cols = ["city", "postcode", "street", "housenumber"]

for df in (osm, gmaps):
    df["_key"] = (
        df[match_cols]
        .astype(str)
        .apply(lambda s: s.str.lower().str.replace(r"[\s/\-.]+", "", regex=True))
        .agg("|".join, axis=1)
    )

In [14]:
#  gmaps-ból ki dobjuk azokat amik már osm-ben vannak
osm_keys = set(osm["_key"])
gmaps_new = gmaps[~gmaps["_key"].isin(osm_keys)].copy()
print("gmaps eredeti:", len(gmaps), "| duplikátumok OSM-mel:", len(gmaps) - len(gmaps_new))

gmaps eredeti: 2190737 | duplikátumok OSM-mel: 333832


In [ ]:
gmaps eredeti: 2190737 | duplikátumok OSM-mel: 322446
gmaps eredeti: 2190737 | duplikátumok OSM-mel: 333832

egyesített: 2491947
egyesített: 2480561

In [15]:
# Egyesítés és takarítás
merged = pd.concat([osm, gmaps_new], ignore_index=True)
merged = merged.drop(columns=["_key"])
merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=osm.crs)

In [16]:
print("egyesített:", len(merged))
merged.head()

egyesített: 2480561


,geometry,city,postcode,street,housenumber,source
0,POINT (18.13766 46.14637),Orfű,7677,Dollár út,1,osm
1,POINT (18.22533 46.06637),Pécs,7623,Indóház tér,1,osm
2,POINT (17.55781 46.7501),Fonyód,8640,Ady Endre utca,2,osm
3,POINT (18.04147 46.7341),Tab,8660,Kossuth Lajos utca,1,osm
4,POINT (17.30675 46.28901),Somogyszob,7563,Keleti köz,1,osm


In [17]:
# minden oszlopo értékének konvertálása azonos típusra export előtt
for col in ["city", "postcode", "street", "housenumber"]:
    merged[col] = merged[col].astype(str)

In [18]:
merged.to_parquet("../../adatok/fix/osm_gmaps_merged.parquet")

In [19]:
# qgis export

# színek megkülönböztetésre
merged["color"] = merged["source"].map({"osm": "#d95f02", "gmaps": "#1b9e77"})

# export
merged.to_file("../../adatok/working/merged_viz.gpkg", driver="GPKG", layer="points")